In [55]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack
import joblib


In [56]:
# df for real news
dfT = pd.read_csv("True.csv")
# df for fake news
dfF = pd.read_csv("Fake.csv")

In [57]:
print(dfF.isnull().sum())
print(dfT.isnull().sum())

title      0
text       0
subject    0
date       0
dtype: int64
title      0
text       0
subject    0
date       0
dtype: int64


In [58]:
dfT["label"] = 1
dfF["label"] = 0

In [59]:
# merging both df
df = pd.concat([dfT,dfF],ignore_index=True)

In [60]:
df.sample(5)

,title,text,subject,date,label
33423,BREAKING: FORD CEO CITES TRUMP In Announcement...,The Government-Orchestrated Bankruptcies Of G...,politics,"Jan 3, 2017",0
30936,WATCH: NAVY VET Declines Award At New Orleans ...,One courageous Navy vet has had enough of the ...,politics,"Nov 3, 2017",0
36600,BREAKING: HUGE LEGAL VICTORY FOR AMERICANS Who...,A big slap in the face to an overreaching gove...,politics,"Sep 22, 2015",0
4178,Trump asks for probe into imports of foreign-m...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 20, 2017",1
1764,Trump administration sued over phone searches ...,WASHINGTON (Reuters) - The Trump administratio...,politicsNews,"September 13, 2017",1


In [61]:
# dropping date column
df = df.drop(["date"],axis=1)

In [62]:
x = df[["title","text","subject"]]
y = df["label"]

In [63]:
# distribution of subject categories in the dataset
x["subject"].value_counts()

politicsNews       11272
worldnews          10145
News                9050
politics            6841
left-news           4459
Government News     1570
US_News              783
Middle-east          778
Name: subject, dtype: int64

In [64]:
x.sample(10)

,title,text,subject
8208,"Trump says Clinton could shoot someone, not be...","PENSACOLA, Fla. (Reuters) - Republican preside...",politicsNews
13693,Japan detects radio signals pointing to possib...,TOKYO/WASHINGTON (Reuters) - Japan has detecte...,worldnews
26615,WATCH: Donald Trump Is OBSESSED With His Desi...,The prospect of Donald Trump becoming presiden...,News
27893,Chris Wallace Gets NC Governor To Admit That ...,Fox host Chris Wallace pinned bigoted North Ca...,News
38996,WALMART Is Selling “Made In Mexico” Apparel Fe...,"The domestic terror group, Antifa, has been ar...",left-news
39144,BOOM! SARAH HUCKABEE SANDERS Sets Media Straig...,Daily Caller The revelation on Monday that f...,left-news
7515,New York City plans largest-ever Election Day ...,NEW YORK (Reuters) - Both U.S. presidential ca...,politicsNews
18622,"Syrian army, allies seize more of Jordanian fr...",BEIRUT (Reuters) - The Syrian army and its all...,worldnews
11351,Syria says military jet downed in northern Ham...,AMMAN (Reuters) - Syria s armed forces said in...,worldnews
33716,GREAT PICK! KT McFarland To Join Team Trump [V...,In asking KT McFarland to become his Deputy N...,politics


In [65]:
from sklearn.preprocessing import OneHotEncoder
subject_encoder = OneHotEncoder(sparse=True, handle_unknown="ignore")
subject_sparse = subject_encoder.fit_transform(x[["subject"]])

In [66]:
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)  # optional: adjust max_features
combined = x["title"]+" - "+x["text"]
X_text = tfidf.fit_transform(combined)

In [67]:
X_final = hstack([X_text, subject_sparse])

In [68]:
x_temp,x_test,y_temp,y_test = train_test_split(X_final,y,test_size=0.2,random_state=42)

In [69]:
x_train,x_val,y_train,y_val = train_test_split(x_temp,y_temp,test_size=0.2,random_state=42)

In [70]:
def evaluate(name,model,x_val,y_val):
    print(f"Evaluation for {name}")
    y_pred=model.predict(x_val);
    print(f"Accuracy is ",accuracy_score(y_val,y_pred))
    print("Classification Report:\n", classification_report(y_val, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))

Logistic Regression

In [71]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(x_train,y_train)

LogisticRegression(max_iter=1000)

In [72]:
evaluate("Logistic Regression",log_model,x_val,y_val)


Evaluation for Logistic Regression
Accuracy is  1.0
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      3773
           1       1.00      1.00      1.00      3411

    accuracy                           1.00      7184
   macro avg       1.00      1.00      1.00      7184
weighted avg       1.00      1.00      1.00      7184

Confusion Matrix:
 [[3773    0]
 [   0 3411]]


Naive Bayes

In [73]:
nb_model = MultinomialNB()
nb_model.fit(x_train,y_train)

MultinomialNB()

In [74]:
evaluate("Naive - Bayes",nb_model,x_val,y_val)


Evaluation for Naive - Bayes
Accuracy is  0.9997216035634744
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      3773
           1       1.00      1.00      1.00      3411

    accuracy                           1.00      7184
   macro avg       1.00      1.00      1.00      7184
weighted avg       1.00      1.00      1.00      7184

Confusion Matrix:
 [[3771    2]
 [   0 3411]]


Random Forest Classifier

In [75]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(x_train, y_train)

RandomForestClassifier(random_state=42)

In [76]:
evaluate("Random Forest",rf_model,x_val,y_val)


Evaluation for Random Forest
Accuracy is  0.9995824053452116
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      3773
           1       1.00      1.00      1.00      3411

    accuracy                           1.00      7184
   macro avg       1.00      1.00      1.00      7184
weighted avg       1.00      1.00      1.00      7184

Confusion Matrix:
 [[3770    3]
 [   0 3411]]


Logistic regression method works better 

In [77]:
y_pred=log_model.predict(x_test);
print(f"Accuracy is ",accuracy_score(y_test,y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy is  1.0
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      4650
           1       1.00      1.00      1.00      4330

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980

Confusion Matrix:
 [[4650    0]
 [   0 4330]]


In [78]:
import joblib

joblib.dump(log_model,"Log_reg_model.pkl")
joblib.dump(tfidf,"tfidf.pkl")
joblib.dump(subject_encoder, "subject_encoder.pkl")


['subject_encoder.pkl']